In [2]:
import pandas as pd 
train_df = pd.read_csv(r"E:\1_Data_Science\Machine-Learning-Projects\9.Building_Machine_Learning_App\Sentimental_Analysis_Project\train data\train.csv")
test_df = pd.read_csv(r"E:\1_Data_Science\Machine-Learning-Projects\9.Building_Machine_Learning_App\Sentimental_Analysis_Project\train data\test.csv")
print("Training data: \n", train_df.head())
print("Test Data: \n", test_df.head())


Training data: 
        textID                                               text  \
0  cb774db0d1                I`d have responded, if I were going   
1  549e992a42      Sooo SAD I will miss you here in San Diego!!!   
2  088c60f138                          my boss is bullying me...   
3  9642c003ef                     what interview! leave me alone   
4  358bd9e861   Sons of ****, why couldn`t they put them on t...   

                         selected_text sentiment  
0  I`d have responded, if I were going   neutral  
1                             Sooo SAD  negative  
2                          bullying me  negative  
3                       leave me alone  negative  
4                        Sons of ****,  negative  
Test Data: 
        textID                                               text sentiment
0  f87dea47db  Last session of the day  http://twitpic.com/67ezh   neutral
1  96d74cb729   Shanghai is also really exciting (precisely -...  positive
2  eee518ae67  Recession hit V

Data Processing 

In [3]:
import re

contractions_dict = {"can`t": "can not",
                     "won`t": "will not",
                     "don`t": "do not",
                     "aren`t": "are not",
                     "i`d": "i would",
                     "couldn`t": "could not",
                     "shouldn`t": "should not",
                     "wouldn`t": "would not",
                     "isn`t": "is not",
                     "it`s": "it is",
                     "didn`t": "did not",
                     "weren`t": "were not",
                     "mustn`t": "must not",
                     }

def prepare_data(df:pd.DataFrame) -> pd.DataFrame:
    df["text"] = (
                df["text"]
                .apply(lambda x: re.split('http://.*', str(x))[0])
                .str.lower()
                .apply(lambda x: replace_words(x, contractions_dict))
    )
    df["label"] = df["sentiment"].map(
                    {"neutral": 1, "negative":0, "positive":2}
                    )
    return df.text.values, df.label.values

def replace_words(string:str, dictionary:dict):
        for k, v in dictionary.items():
                string = string.replace(k, v)
        return string

train_tweets, train_labels = prepare_data(train_df)
test_tweets, test_labels = prepare_data(test_df)

Tokenization

 tweet into a single fixed-length vector – specifically a TFIDF integration

 To do this we can use Tokenizer() built into Keras, suitable for training data

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_tweets)

train_tokenized = tokenizer.texts_to_matrix(train_tweets, mode='tfidf')
test_tokenized = tokenizer.texts_to_matrix(test_tweets, mode='tfidf')

Machine learning model for Sentiment Analysis 

In [6]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(n_estimators=500, min_samples_leaf=2, oob_score=True, n_jobs=-1,random_state=42)
forest.fit(train_tokenized, train_labels)
print(f"Train_score: {forest.score(train_tokenized, train_labels)}")
print(f"OOB score: {forest.oob_score_}")

Train_score: 0.7658018267166405
OOB score: 0.6854190167752265


In [7]:
print("Tet score: ", forest.score(test_tokenized, test_labels))

Tet score:  0.6870401810979061


In [9]:
import pickle

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save random forest model
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(forest, f)
